# 4. Restore your photos

Restores photos with the **lightweight direct mapping** and, for comparison, with **BOPBTL's original mapping**. Both use the same frozen VAEs and the same pre- and post-processing.

No Google Drive needed: the model weights are downloaded from the GitHub release.

**Steps:**
1. Run the setup and code cells.
2. Put your photos in `INPUT_DIR`.
3. Run the last cell.

**Output:** the raw outputs, plus a side-by-side image per photo. `GUARD` is optional post-processing; when on, it is applied to every method in the same way.

## Setup

In [ ]:
import os, subprocess, sys
if not os.path.isdir("/content/Bringing-Old-Photos-Back-to-Life"):
    subprocess.run(["git", "clone", "-q", "https://github.com/microsoft/Bringing-Old-Photos-Back-to-Life",
                    "/content/Bringing-Old-Photos-Back-to-Life"], check=True)
sys.path.insert(0, "/content")

## Code (identical to `src/`)

### `old_photo_model.py`

In [ ]:
%%writefile /content/old_photo_model.py
"""
OldPhotoModel: single-class wrapper around microsoft/Bringing-Old-Photos-Back-to-Life
(Global restoration stage). Finds the repo (Colab: /content/...), downloads the pretrained
weights if missing, builds VAE A, VAE B and the mapping network, loads their weights and
verifies that every checkpoint key matched.

    from old_photo_model import OldPhotoModel
    obj = OldPhotoModel(mode="quality", gpu=0)

    x   = obj.to_tensor(PIL_image)
    z_A = obj.encode_A(x)
    z_B = obj.map(z_A)
    y   = obj.decode_B(z_B)            # or obj.translate(x)
    obj.to_image(y).save("out.png")
    obj.restore(PIL_image).save("out.png")

Modes: "quality", "scratch" (needs mask), "scratch_hr" (needs mask, high-res).
Scratch modes: x, m = obj.prepare(image, mask); obj.map(z_A, m) / obj.translate(x, m).
"""
import contextlib
import glob
import os
import shutil
import sys
import tempfile
import urllib.request
import zipfile

import numpy as np
import torch
from PIL import Image, ImageFilter

_REPO_ZIP = "https://github.com/microsoft/Bringing-Old-Photos-Back-to-Life/archive/refs/heads/master.zip"
_CKPT_ZIP = ("https://github.com/microsoft/Bringing-Old-Photos-Back-to-Life/"
             "releases/download/v1.0/global_checkpoints.zip")
_DEFAULT_ROOT = "/content/Bringing-Old-Photos-Back-to-Life"

# (checkpoint sub-folder, network label) required per mode -> files named "<epoch>_net_<label>.pth"
_REQUIRED = {
    "quality":    [("VAE_A_quality", "G"), ("VAE_B_quality", "G"), ("mapping_quality", "mapping_net")],
    "scratch":    [("VAE_A_quality", "G"), ("VAE_B_scratch", "G"), ("mapping_scratch", "mapping_net")],
    "scratch_hr": [("VAE_A_quality", "G"), ("VAE_B_scratch", "G"), ("mapping_Patch_Attention", "mapping_net")],
}


def _download(url, dest):
    print(f"[OldPhotoModel] downloading {url}")
    with urllib.request.urlopen(url) as r, open(dest, "wb") as f:
        shutil.copyfileobj(r, f)


def _find_repo(hint=None):
    """Return repo root: hint, /content/<repo>, or any folder under /content containing Global/models/mapping_model.py."""
    marker = os.path.join("Global", "models", "mapping_model.py")
    cands = [hint, _DEFAULT_ROOT, _DEFAULT_ROOT + "-master"]
    cands += glob.glob("/content/*/") + glob.glob("/content/*/*/")
    for c in cands:
        if c and os.path.isfile(os.path.join(c, marker)):
            return os.path.abspath(c)
    return None


@contextlib.contextmanager
def _cpu_safe_torch_load():
    """The repo calls torch.load(path) without map_location, which fails on CPU-only machines."""
    orig = torch.load

    def patched(*args, **kwargs):
        kwargs.setdefault("map_location", "cpu")
        return orig(*args, **kwargs)

    torch.load = patched
    try:
        yield
    finally:
        torch.load = orig


class OldPhotoModel:

    def __init__(self, repo_root=None, mode="quality", gpu=0, auto_download=True, verify=True,
                 test_mode="Full"):
        assert mode in _REQUIRED, f"mode must be one of {list(_REQUIRED)}"
        assert test_mode in ("Full", "Scale"), "test_mode: 'Full' (keep size) or 'Scale' (short side -> 256)"
        self.mode = mode
        self.test_mode = test_mode  # only used in quality mode, like --test_mode in Global/test.py

        found = _find_repo(repo_root)
        if found is None:
            if not auto_download:
                raise FileNotFoundError("Repo not found under /content; pass repo_root=...")
            repo_root = os.path.abspath(repo_root or _DEFAULT_ROOT)
            self._ensure_repo(repo_root)  # zip download as fallback
        else:
            repo_root = found
        global_dir = os.path.join(repo_root, "Global")
        print(f"[OldPhotoModel] using repo at {repo_root}")

        if auto_download:
            self._ensure_weights(global_dir, mode)
        else:
            self._check_weights(global_dir, mode, raise_error=True)

        if gpu >= 0 and not torch.cuda.is_available():
            print("[OldPhotoModel] CUDA not available -> running on CPU")
            gpu = -1

        opt = self._build_opt(global_dir, mode, gpu)

        from models.mapping_model import Pix2PixHDModel_Mapping  # repo code

        prev = os.getcwd()
        os.chdir(global_dir)
        try:
            with _cpu_safe_torch_load():
                core = Pix2PixHDModel_Mapping()
                core.initialize(opt)  # builds netG_A, netG_B, mapping_net + loads weights
        finally:
            os.chdir(prev)

        self.vae_A = core.netG_A      # old-photo domain VAE
        self.vae_B = core.netG_B      # clean domain VAE
        self.mapping = core.mapping_net
        for net in (self.vae_A, self.vae_B, self.mapping):
            net.eval()
            for p in net.parameters():
                p.requires_grad = False

        self.uses_mask = bool(getattr(opt, "NL_use_mask", False))
        self._hr = bool(getattr(opt, "inference_optimize", False))
        self._mask_dilation = int(getattr(opt, "mask_dilation", 0) or 0)
        self.device = next(self.mapping.parameters()).device
        self._core = core
        self._opt = opt

        if verify:
            self._verify_weights(opt, mode)

    # ------------------------------------------------------------------ #
    # Core methods
    # ------------------------------------------------------------------ #
    @torch.no_grad()
    def encode_A(self, x):
        return self.vae_A.forward(x, flow="enc")

    @torch.no_grad()
    def encode_B(self, x):
        return self.vae_B.forward(x, flow="enc")

    @torch.no_grad()
    def map(self, z_A, mask=None):
        if not self.uses_mask:
            return self.mapping(z_A)
        if mask is None:
            raise ValueError("scratch mode: pass mask (1,1,H,W) tensor, same H/W as the input image "
                             "(use obj.prepare(image, mask)).")
        if self._hr:
            return self.mapping.inference_forward(z_A, mask)
        return self.mapping(z_A, mask)

    @torch.no_grad()
    def decode_A(self, z_A):
        return self.vae_A.forward(z_A, flow="dec")

    @torch.no_grad()
    def decode_B(self, z_B):
        return self.vae_B.forward(z_B, flow="dec")

    @torch.no_grad()
    def translate(self, x_A, mask=None):
        z_A = self.encode_A(x_A)
        z_B = self.map(z_A, mask)
        return self.decode_B(z_B)

    # ------------------------------------------------------------------ #
    # Helpers: PIL <-> tensor, and one-call restore
    # ------------------------------------------------------------------ #
    def _fit(self, img):
        """Same resizing as Global/test.py (BILINEAR, multiples of 4).
        Scratch modes never rescale in test.py, so only the multiple-of-4 rounding applies there."""
        w, h = img.size
        ow, oh = w, h
        if self.test_mode == "Scale" and not self.uses_mask:
            if w < h:
                ow, oh = 256, h / w * 256
            else:
                oh, ow = 256, w / h * 256
        size = (int(round(ow / 4) * 4), int(round(oh / 4) * 4))
        return img if size == (w, h) else img.resize(size, Image.BILINEAR)

    def to_tensor(self, image):
        """PIL -> (1,3,H,W) tensor in [-1,1]; H,W rounded to multiples of 4."""
        image = self._fit(image.convert("RGB"))
        x = torch.from_numpy(np.asarray(image, dtype=np.float32) / 255.0).permute(2, 0, 1)
        return ((x - 0.5) / 0.5).unsqueeze(0).to(self.device)

    def mask_to_tensor(self, mask, size):
        """PIL mask (white = scratch) -> (1,1,H,W) tensor, resized to size=(W,H)."""
        m = mask.convert("RGB").resize(size, Image.NEAREST)
        m = torch.from_numpy(np.asarray(m, dtype=np.float32) / 255.0).permute(2, 0, 1)
        return m[:1].unsqueeze(0).to(self.device)

    def prepare(self, image, mask=None):
        """PIL (+ PIL mask in scratch modes) -> (x, mask_tensor or None), preprocessed like Global/test.py:
        size handled like test.py, mask dilated (HR mode), scratched pixels painted white."""
        img = self._fit(image.convert("RGB"))
        if not self.uses_mask:
            return self.to_tensor(img), None
        if mask is None:
            raise ValueError("scratch mode needs a mask image (white = scratch).")
        mk = mask.convert("RGB").resize(img.size, Image.NEAREST)
        for _ in range(self._mask_dilation):
            mk = mk.filter(ImageFilter.MaxFilter(3))
        m = np.asarray(mk, dtype=np.float32) / 255.0
        holed = np.asarray(img, dtype=np.float32) * (1.0 - m) + 255.0 * m
        img = Image.fromarray(holed.round().clip(0, 255).astype(np.uint8))
        return self.to_tensor(img), self.mask_to_tensor(mk, img.size)

    @staticmethod
    def to_image(x, normalize=True):
        """(1,3,H,W) tensor in [-1,1] -> PIL. normalize=True min-max stretches like
        save_image(..., normalize=True) in Global/test.py; False = plain clamp."""
        y = (x.detach().cpu().float() + 1.0) / 2.0
        if normalize:
            lo, hi = y.min(), y.max()
            y = (y - lo) / (hi - lo).clamp_min(1e-5)
        y = y.clamp(0, 1)[0]
        return Image.fromarray((y.permute(1, 2, 0).numpy() * 255).round().astype(np.uint8))

    def restore(self, image, mask=None, keep_size=True, normalize=True):
        """PIL in -> PIL out (full pipeline)."""
        x, m = self.prepare(image, mask)
        out = self.to_image(self.translate(x, m), normalize)
        if keep_size and out.size != image.size:
            out = out.resize(image.size, Image.BICUBIC)
        return out

    # ------------------------------------------------------------------ #
    # Setup: repo, weights, options
    # ------------------------------------------------------------------ #
    @staticmethod
    def _ensure_repo(repo_root):
        if os.path.isfile(os.path.join(repo_root, "Global", "models", "mapping_model.py")):
            return
        os.makedirs(os.path.dirname(repo_root), exist_ok=True)
        with tempfile.TemporaryDirectory() as tmp:
            zpath = os.path.join(tmp, "repo.zip")
            _download(_REPO_ZIP, zpath)
            with zipfile.ZipFile(zpath) as z:
                z.extractall(tmp)
            extracted = os.path.join(tmp, "Bringing-Old-Photos-Back-to-Life-master")
            if os.path.exists(repo_root):
                shutil.rmtree(repo_root)
            shutil.move(extracted, repo_root)

    def _ensure_weights(self, global_dir, mode):
        """Download the checkpoint zip (layout: checkpoints/restoration/<folder>/*.pth) and copy
        any missing or empty files for the needed folders."""
        if not self._missing(global_dir, mode):
            return
        target = os.path.join(global_dir, "checkpoints", "restoration")
        wanted = {f for f, _ in _REQUIRED[mode]}
        with tempfile.TemporaryDirectory() as tmp:
            zpath = os.path.join(tmp, "global_checkpoints.zip")
            _download(_CKPT_ZIP, zpath)
            extract_dir = os.path.join(tmp, "extracted")
            with zipfile.ZipFile(zpath) as z:
                z.extractall(extract_dir)
            src_root = os.path.join(extract_dir, "checkpoints", "restoration")
            for folder in wanted:
                os.makedirs(os.path.join(target, folder), exist_ok=True)
                for f in os.listdir(os.path.join(src_root, folder)):
                    src, dst = os.path.join(src_root, folder, f), os.path.join(target, folder, f)
                    if not os.path.isfile(dst) or os.path.getsize(dst) != os.path.getsize(src):
                        shutil.copy2(src, dst)
        self._check_weights(global_dir, mode, raise_error=True)

    @staticmethod
    def _missing(global_dir, mode):
        ckpt = os.path.join(global_dir, "checkpoints", "restoration")
        missing = []
        for folder, label in _REQUIRED[mode]:
            if not glob.glob(os.path.join(ckpt, folder, f"*_net_{label}.pth")):
                missing.append(os.path.join(ckpt, folder, f"latest_net_{label}.pth"))
        return missing

    def _check_weights(self, global_dir, mode, raise_error):
        missing = self._missing(global_dir, mode)
        if missing and raise_error:
            raise FileNotFoundError("Missing checkpoints:\n  " + "\n  ".join(missing) +
                                    f"\nDownload {_CKPT_ZIP} and unzip it inside {global_dir}")
        return missing

    @staticmethod
    def _build_opt(global_dir, mode, gpu):
        if global_dir not in sys.path:
            sys.path.insert(0, global_dir)  # repo imports `options`, `models`, `util` from Global/
        from options.test_options import TestOptions

        flag = "--Quality_restore" if mode == "quality" else "--Scratch_and_Quality_restore"
        backup = sys.argv
        sys.argv = ["old_photo_model", flag, "--gpu_ids", str(gpu)]
        if mode == "scratch_hr":
            sys.argv.append("--HR")
        try:
            opt = TestOptions().parse(save=False)
        finally:
            sys.argv = backup

        # mirrors parameter_set() in Global/test.py
        ckpt = os.path.join(global_dir, "checkpoints", "restoration")
        opt.serial_batches = True
        opt.no_flip = True
        opt.label_nc = 0
        opt.n_downsample_global = 3
        opt.mc = 64
        opt.k_size = 4
        opt.start_r = 1
        opt.mapping_n_block = 6
        opt.map_mc = 512
        opt.no_instance = True
        opt.checkpoints_dir = ckpt

        if mode == "quality":
            opt.name = "mapping_quality"
            opt.load_pretrainA = os.path.join(ckpt, "VAE_A_quality")
            opt.load_pretrainB = os.path.join(ckpt, "VAE_B_quality")
        else:
            opt.NL_res = True
            opt.use_SN = True
            opt.correlation_renormalize = True
            opt.NL_use_mask = True
            opt.NL_fusion_method = "combine"
            opt.non_local = "Setting_42"
            opt.name = "mapping_scratch"
            opt.load_pretrainA = os.path.join(ckpt, "VAE_A_quality")
            opt.load_pretrainB = os.path.join(ckpt, "VAE_B_scratch")
            if mode == "scratch_hr":
                opt.HR = True
                opt.mapping_exp = 1              # Mapping_Model_with_mask_2 (patch attention)
                opt.inference_optimize = True    # enables mapping.inference_forward
                opt.mask_dilation = 3
                opt.name = "mapping_Patch_Attention"
        return opt

    def _verify_weights(self, opt, mode):
        """Re-load each checkpoint with strict=True so a silent partial load can't go unnoticed."""
        ckpt = opt.checkpoints_dir
        vae_epoch = getattr(opt, "use_vae_which_epoch", "latest")
        map_epoch = getattr(opt, "which_epoch", "latest")
        jobs = [
            (self.vae_A, os.path.join(opt.load_pretrainA, f"{vae_epoch}_net_G.pth")),
            (self.vae_B, os.path.join(opt.load_pretrainB, f"{vae_epoch}_net_G.pth")),
            (self.mapping, os.path.join(ckpt, opt.name, f"{map_epoch}_net_mapping_net.pth")),
        ]
        for net, path in jobs:
            try:
                net.load_state_dict(torch.load(path, map_location="cpu"), strict=True)
            except Exception as e:  # noqa: BLE001
                raise RuntimeError(
                    f"Checkpoint {path} does not match the network built for mode '{mode}': {e}\n"
                    "The option flags in _build_opt() probably differ from Global/test.py."
                ) from e

### `direct_model.py`

In [ ]:
%%writefile /content/direct_model.py
"""
DirectMapping: the final z_A -> z_B mapping, a plain residual conv network in the
frozen VAE latent space (same mechanism as Microsoft's mapping network, ~3x smaller).

It is rf_model.VelocityNet with the time input removed. With t fixed to 0:
  - the time embedding is a constant vector, so each FiLM layer is a constant
    per-channel scale/shift right after norm2, which norm2's own affine absorbs;
  - the input concat(z_t, z_A) is concat(z_A, z_A), so the first conv's two weight
    halves merge into one conv on z_A.
So the function class is exactly that of the t = 0 model, and fold_rf() turns a
trained t = 0 RFMapping into a DirectMapping with identical outputs.

    z_B = scale * (a + net(a)),   a = z_A / scale

Residual, with a zero-initialized output conv, so training starts at the identity.
forward(z_A, mask=None) matches OldPhotoModel.mapping's call signature: obj.mapping = net.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F


class ResBlock(nn.Module):
    """FiLMResBlock without FiLM: GroupNorm -> SiLU -> dilated conv -> GroupNorm -> SiLU -> dropout -> conv, + skip."""

    def __init__(self, channels, dilation=1, dropout=0.1):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, channels)
        self.norm2 = nn.GroupNorm(8, channels)
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=dilation, dilation=dilation)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.dropout = nn.Dropout2d(dropout)

    def forward(self, x):
        h = self.conv1(F.silu(self.norm1(x)))
        h = self.dropout(F.silu(self.norm2(h)))
        return x + self.conv2(h)


class DirectMapping(nn.Module):
    def __init__(self, latent_ch, width=256, n_blocks=9, dropout=0.1):
        super().__init__()
        self.latent_ch, self.width, self.n_blocks = latent_ch, width, n_blocks
        self.in_conv = nn.Conv2d(latent_ch, width, 3, padding=1)
        dilations = (1, 2, 4)
        self.blocks = nn.ModuleList([ResBlock(width, dilations[i % len(dilations)], dropout)
                                     for i in range(n_blocks)])
        self.out_conv = nn.Conv2d(width, latent_ch, 3, padding=1)
        nn.init.zeros_(self.out_conv.weight)
        nn.init.zeros_(self.out_conv.bias)
        self.register_buffer("scale", torch.ones(()))  # latent std, set from training data

    def forward(self, z_A, mask=None):
        a = z_A / self.scale
        h = self.in_conv(a)
        for block in self.blocks:
            h = block(h)
        return (a + self.out_conv(F.silu(h))) * self.scale

    def arch(self):
        return {"type": "direct", "latent_ch": self.latent_ch, "width": self.width, "n_blocks": self.n_blocks}


@torch.no_grad()
def fold_rf(rf):
    """t = 0 RFMapping -> DirectMapping with identical outputs (both in eval mode)."""
    v, c = rf.v, rf.latent_ch
    d = DirectMapping(c, rf.width, rf.n_blocks).to(rf.scale.device)
    d.scale.copy_(rf.scale)
    w = v.in_conv.weight                                  # (width, 2c, 3, 3): halves for z_t and z_A
    d.in_conv.weight.copy_(w[:, :c] + w[:, c:])
    d.in_conv.bias.copy_(v.in_conv.bias)
    t_emb = v.time_emb(torch.zeros(1, device=w.device))   # constant at t = 0
    for bd, bv in zip(d.blocks, v.blocks):
        scale, shift = bv.film(t_emb)[0].chunk(2)
        for name in ("norm1", "conv1", "conv2"):
            getattr(bd, name).load_state_dict(getattr(bv, name).state_dict())
        # norm2(h) * (1 + scale) + shift = gn(h) * w (1 + scale) + (b (1 + scale) + shift)
        bd.norm2.weight.copy_(bv.norm2.weight * (1 + scale))
        bd.norm2.bias.copy_(bv.norm2.bias * (1 + scale) + shift)
    d.out_conv.load_state_dict(v.out_conv.state_dict())
    return d.eval()


def load_direct(path, device, use_ema=True):
    """DirectMapping from a train_direct checkpoint (or the exported weights file). Returns (net, step)."""
    ck = torch.load(path, map_location=device)
    arch = {k: v for k, v in ck["arch"].items() if k != "type"}
    net = DirectMapping(**arch).to(device)
    net.load_state_dict(ck["ema"] if use_ema else ck["net"])
    net.eval()
    print(f"loaded {path} | step {ck['step']} | {sum(p.numel() for p in net.parameters()) / 1e6:.2f}M params")
    return net, ck["step"]

## 4. Restore

| Setting | Meaning |
|---|---|
| `WEIGHTS` | The final model; downloaded from the v1.0 release if the file is missing. Any `final.pt` from notebook 2 also works |
| `METHODS` | Remove `"Microsoft"` to restore with ours only |
| `GUARD` | `None` (raw, default); `"gray"`: outputs of gray inputs become gray; `"near_gray"`: low-saturation inputs keep the output's lightness and the input's smoothed colour. **Applied to all methods equally** |
| `MAX_SIDE` | Downscales large photos (GPU memory) |

In [ ]:
import glob
import os

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from direct_model import load_direct
from old_photo_model import OldPhotoModel

WEIGHTS = "/content/mapping_direct.pt"       # downloaded from the release if missing
WEIGHTS_URL = "https://github.com/Mani-Sankar-d/lightweight-old-photo-restoration/releases/download/v1.0/mapping_direct.pt"
INPUT_DIR = "/content/inputs"                 # put your photos here
OUTPUT_DIR = "/content/outputs"
METHODS = ["Ours", "Microsoft"]
GUARD = None                                  # None | "gray" | "near_gray"  (applied to every method)
MAX_SIDE = 1024
IMG_EXT = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp")
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(WEIGHTS):
    torch.hub.download_url_to_file(WEIGHTS_URL, WEIGHTS)
if "obj" not in globals():
    obj = OldPhotoModel(mode="quality", gpu=0)
ours, _ = load_direct(WEIGHTS, obj.device)
nets = {"Ours": ours, "Microsoft": obj._core.mapping_net}


def restore(img, method):
    """Raw output of one method; identical VAEs and pre/post-processing for all methods."""
    ms_mapping, ms_mask = obj.mapping, obj.uses_mask
    obj.mapping, obj.uses_mask = nets[method], False
    try:
        with torch.no_grad():
            return obj.restore(img, keep_size=True, normalize=False)
    finally:
        obj.mapping, obj.uses_mask = ms_mapping, ms_mask


def mean_chroma(img):
    a = np.asarray(img.convert("RGB"), dtype=np.float32)
    return float(np.abs(a - a.mean(axis=2, keepdims=True)).mean())


def guard(inp, out):
    """Optional post-processing, applied to every method's output in the same way."""
    if GUARD == "gray" and mean_chroma(inp) <= 2.0:
        return out.convert("L").convert("RGB")
    if GUARD == "near_gray" and mean_chroma(inp) <= 12.0:
        lab_in = cv2.cvtColor(np.asarray(inp.convert("RGB").resize(out.size)), cv2.COLOR_RGB2LAB)
        lab_out = cv2.cvtColor(np.asarray(out.convert("RGB")), cv2.COLOR_RGB2LAB)
        ab = cv2.GaussianBlur(lab_in[..., 1:], (31, 31), 0)
        return Image.fromarray(cv2.cvtColor(np.dstack([lab_out[..., :1], ab]), cv2.COLOR_LAB2RGB))
    return out


paths = sorted(p for p in glob.glob(os.path.join(INPUT_DIR, "*")) if p.lower().endswith(IMG_EXT))
assert paths, f"no images in {INPUT_DIR}: upload some first"
for p in paths:
    name = os.path.splitext(os.path.basename(p))[0]
    img = Image.open(p).convert("RGB")
    if MAX_SIDE and max(img.size) > MAX_SIDE:
        s = MAX_SIDE / max(img.size)
        img = img.resize((round(img.size[0] * s), round(img.size[1] * s)), Image.BICUBIC)
    outs = {"Input": img}
    for m in METHODS:
        outs[m] = guard(img, restore(img, m))
        outs[m].save(f"{OUTPUT_DIR}/{name}_{m}.png")
    fig, axes = plt.subplots(1, len(outs), figsize=(5 * len(outs), 5 * img.size[1] / img.size[0] + 0.5), squeeze=False)
    for ax, (k, v) in zip(axes[0], outs.items()):
        ax.imshow(v)
        ax.set_title(k + (f" (guard: {GUARD})" if GUARD and k != "Input" else ""))
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/{name}_compare.png", dpi=120, bbox_inches="tight")
    plt.show()
print(f"{len(paths)} photos restored -> {OUTPUT_DIR}")